In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings 
warnings.filterwarnings('ignore')


In [2]:
df=pd.read_csv('UCI_Credit_Card.csv')
df.head(2)

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1


In [3]:
df['default.payment.next.month'].value_counts()
#This shows that the dataset is severely imbalanced

default.payment.next.month
0    23364
1     6636
Name: count, dtype: int64

In [ ]:
df.drop('ID',axis=1,inplace=True) 

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   LIMIT_BAL                   30000 non-null  float64
 1   SEX                         30000 non-null  int64  
 2   EDUCATION                   30000 non-null  int64  
 3   MARRIAGE                    30000 non-null  int64  
 4   AGE                         30000 non-null  int64  
 5   PAY_0                       30000 non-null  int64  
 6   PAY_2                       30000 non-null  int64  
 7   PAY_3                       30000 non-null  int64  
 8   PAY_4                       30000 non-null  int64  
 9   PAY_5                       30000 non-null  int64  
 10  PAY_6                       30000 non-null  int64  
 11  BILL_AMT1                   30000 non-null  float64
 12  BILL_AMT2                   30000 non-null  float64
 13  BILL_AMT3                   300

In [6]:
!pip install xgboost

   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 1.3/150.0 MB 7.4 MB/s eta 0:00:20
    --------------------------------------- 2.6/150.0 MB 6.6 MB/s eta 0:00:23
   - -------------------------------------- 3.9/150.0 MB 6.7 MB/s eta 0:00:22
   - -------------------------------------- 5.2/150.0 MB 6.2 MB/s eta 0:00:24
   - -------------------------------------- 6.0/150.0 MB 6.1 MB/s eta 0:00:24
   -- ------------------------------------- 7.6/150.0 MB 6.1 MB/s eta 0:00:24
   -- ------------------------------------- 9.2/150.0 MB 6.4 MB/s eta 0:00:22
   -- ------------------------------------- 11.0/150.0 MB 6.7 MB/s eta 0:00:21
   --- ------------------------------------ 12.8/150.0 MB 6.9 MB/s eta 0:00:20
   --- ------------------------------------ 14.4/150.0 MB 7.0 MB/s eta 0:00:20
   ---- ----------------------------------- 16.3/150.0 MB 7.2 MB/s eta 0:00:19
   ---- ----------------------------------- 17.8/150.0 MB 7.3 MB/s 

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,f1_score,precision_score,recall_score,roc_auc_score

In [8]:
for col in ['SEX','EDUCATION','MARRIAGE','PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']:
    df[col]=df[col].astype('category')

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   LIMIT_BAL                   30000 non-null  float64 
 1   SEX                         30000 non-null  category
 2   EDUCATION                   30000 non-null  category
 3   MARRIAGE                    30000 non-null  category
 4   AGE                         30000 non-null  int64   
 5   PAY_0                       30000 non-null  category
 6   PAY_2                       30000 non-null  category
 7   PAY_3                       30000 non-null  category
 8   PAY_4                       30000 non-null  category
 9   PAY_5                       30000 non-null  category
 10  PAY_6                       30000 non-null  category
 11  BILL_AMT1                   30000 non-null  float64 
 12  BILL_AMT2                   30000 non-null  float64 
 13  BILL_AMT3       

In [ ]:
x=df.drop('default.payment.next.month',axis=1) #Independent Features
y=df['default.payment.next.month']             #Dependent Feature

In [11]:
cat_features = x.select_dtypes(include=['object', 'category']).columns.tolist()
num_features1 = x.select_dtypes(include=['int64', 'float64']).columns.tolist()

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
         ("OneHotEncoder", oh_transformer, cat_features),
          ("StandardScaler", numeric_transformer, num_features1)
    ]
)

In [ ]:
x_scaled=preprocessor.fit_transform(x) #transforming the data

In [13]:
#Creating training and test dataset
x_train,x_test,y_train,y_test=train_test_split(x_scaled,y,test_size=0.2,random_state=42)
x_train.shape,y_test.shape

((24000, 82), (6000,))

In [14]:
lr=LogisticRegression()
rf=RandomForestClassifier()
adb=AdaBoostClassifier()
xgb=XGBClassifier()
sv=SVC()

In [15]:
models={
    "Logisitic Regression":LogisticRegression(),
    "AdaBoost Classifier":AdaBoostClassifier(),
    "Random Forest":RandomForestClassifier(),
    "XGBOOST Classifier":XGBClassifier(),
    "Support Vector Machine":SVC()
}
for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(x_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    # Training set performance
    model_train_accuracy = accuracy_score(y_train, y_train_pred) # Calculate Accuracy
    model_train_f1 = f1_score(y_train, y_train_pred, average='weighted') # Calculate F1-score
    model_train_precision = precision_score(y_train, y_train_pred) # Calculate Precision
    model_train_recall = recall_score(y_train, y_train_pred) # Calculate Recall
    model_train_rocauc_score = roc_auc_score(y_train, y_train_pred)


    # Test set performance
    model_test_accuracy = accuracy_score(y_test, y_test_pred) # Calculate Accuracy
    model_test_f1 = f1_score(y_test, y_test_pred, average='weighted') # Calculate F1-score
    model_test_precision = precision_score(y_test, y_test_pred) # Calculate Precision
    model_test_recall = recall_score(y_test, y_test_pred) # Calculate Recall
    model_test_rocauc_score = roc_auc_score(y_test, y_test_pred) #Calculate Roc


    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Accuracy: {:.4f}".format(model_train_accuracy))
    print('- F1 score: {:.4f}'.format(model_train_f1))
    
    print('- Precision: {:.4f}'.format(model_train_precision))
    print('- Recall: {:.4f}'.format(model_train_recall))
    print('- Roc Auc Score: {:.4f}'.format(model_train_rocauc_score))

    
    
    print('----------------------------------')
    
    print('Model performance for Test set')
    print('- Accuracy: {:.4f}'.format(model_test_accuracy))
    print('- F1 score: {:.4f}'.format(model_test_f1))
    print('- Precision: {:.4f}'.format(model_test_precision))
    print('- Recall: {:.4f}'.format(model_test_recall))
    print('- Roc Auc Score: {:.4f}'.format(model_test_rocauc_score))

    
    print('='*35)
    print('\n')

Logisitic Regression
Model performance for Training set
- Accuracy: 0.8220
- F1 score: 0.7999
- Precision: 0.6888
- Recall: 0.3605
- Roc Auc Score: 0.6570
----------------------------------
Model performance for Test set
- Accuracy: 0.8193
- F1 score: 0.7964
- Precision: 0.6681
- Recall: 0.3465
- Roc Auc Score: 0.6492


AdaBoost Classifier
Model performance for Training set
- Accuracy: 0.8193
- F1 score: 0.7947
- Precision: 0.6875
- Recall: 0.3393
- Roc Auc Score: 0.6477
----------------------------------
Model performance for Test set
- Accuracy: 0.8182
- F1 score: 0.7924
- Precision: 0.6762
- Recall: 0.3244
- Roc Auc Score: 0.6405


Random Forest
Model performance for Training set
- Accuracy: 0.9995
- F1 score: 0.9995
- Precision: 0.9991
- Recall: 0.9985
- Roc Auc Score: 0.9991
----------------------------------
Model performance for Test set
- Accuracy: 0.8177
- F1 score: 0.7970
- Precision: 0.6498
- Recall: 0.3618
- Roc Auc Score: 0.6536


XGBOOST Classifier
Model performance for T

In [ ]:
#As the dataset is imbalanced , we get a low RECALL value for all models